# API로 데이터 수집

In [ ]:

######## 페이지네이션 적용 코드 ##############

import requests
import os
from datetime import datetime

# API URL 및 쿼리 파라미터
url = "https://gnip-api.twitter.com/search/fullarchive/accounts/ArsPraxia/prod.json"
keyword = 'Korea'  ######## 원하는 키워드로 수정 필요
minus_keyword = ''

###### !!확인 필!! >>> minus_keyword가 잘 들어갔는지 확인 <<< ######
params = {
    "query": (keyword) + " " + (minus_keyword) + " lang:id -is:retweet -is:nullcast -has:links",
    "maxResults": 500,
    "fromDate" : '202301010000',
    "toDate" : '202309302359'
}

# 현재 날짜를 YYYYMMDD 형식의 문자열로 변환
current_date = datetime.now().strftime('%Y%m%d')

# 사용자 인증 정보
auth = ("#####", "#####")



# 폴더 이름 지정
folder_name = f'원시데이터/{current_date}_data/{keyword}'
# folder_name = '날짜테스트'

# 폴더가 없다면 생성
if not os.path.exists(folder_name):
    os.makedirs(folder_name)



# 페이지 번호 초기화
page_number = 1

# 첫 번째 페이지 요청
response = requests.get(url, params=params, auth=auth)

# 응답 확인 및 데이터 저장
while response.status_code == 200:
    # 파일 경로 생성
    file_path = os.path.join(folder_name, f"{current_date}_twitter_{keyword}_{page_number}.json")
    
    # JSON 데이터 저장
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(response.text)
    
    # 응답 JSON 데이터 로드
    response_data = response.json()
    
    # 'next' 키가 있는지 확인하고, 있다면 다음 페이지 데이터 요청
    next_token = response_data.get('next', None)
    if next_token is not None:
        params['next'] = next_token
        response = requests.get(url, params=params, auth=auth)
        page_number += 1  # 페이지 번호 증가
    else:  # 'next' 키가 없다면 루프 종료
        break
else:
    print(f"Error: Unable to fetch data. Status code: {response.status_code}")


<br>
<br>
<br>

# CSV 만들기

In [ ]:
from datetime import datetime


def to_date(date_str):
    # 문자열을 datetime 객체로 변환
    date_obj = datetime.strptime(date_str, "%a %b %d %H:%M:%S +0000 %Y")

    # datetime 객체를 원하는 형식의 문자열로 변환
    formatted_date_str = date_obj.strftime("%Y-%m-%d")
    return formatted_date_str

In [ ]:
############# Version 3 ##################
######## 원하는 형태로 만드는 코드 ############
######## 키워드_assemble로 저장되는 코드 ##############

import pandas as pd
import json
import os
from datetime import datetime

# 모든 JSON 파일을 불러오기 위해 폴더 지정
folder_name = f'원시데이터/{current_date}_data/{keyword}'

# 최종 데이터를 저장할 딕셔너리 초기화
final_data = []


# 폴더 내의 모든 JSON 파일 처리
for file_name in os.listdir(folder_name):

    if file_name.endswith('.json'):
        file_path = os.path.join(folder_name, file_name)
        
        # 파일 읽기
        with open(file_path, 'r', encoding='utf-8') as file:
            json_data = json.load(file)
        date = file_name.split('_')[0]
        source = file_name.split('_')[1]
        keyword=file_name.split('_')[2]
        # 각 결과에 대해 원하는 데이터 추출

    
        for result in json_data['results']:
            # extended_tweet의 full_text 값이 있으면 사용, 없으면 text 값을 사용
            text = result.get('extended_tweet', {}).get('full_text', result['text'])
            created_at = str(result['created_at'])
            

            # 최종 데이터에 추가
            final_data.append({
                'created_at': created_at,
                'Text': text,
                'Filename' :"C:/Users/USER/Desktop/" + file_path,     #"C:/Users/USER/Desktop/트위터데이터/" 가 나으려나
            })

    
keyword = file_name.replace('.json', '').split('_')[2]
df= pd.DataFrame(final_data)
df['Pub_Subj']= keyword
df['Col_Date']=  datetime.now().strftime('%Y-%m-%d')
df['Title']= 'Twitter_'+keyword
df['Doc_ID'] = date + '_' + source + '_' + keyword + df.index.map(lambda x: f"{x + 1:06}")

# 문서 ID, 수집일, 수집 키워드 표기
df['Pub_Date']=df['created_at'].apply(to_date)
df.drop('created_at', axis=1, inplace=True)

df.tail(10)


<br>
<br>
<br>

# 텍스트 정제

In [ ]:
import re

def clean_text(text):
    # 이모티콘 제거
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F700-\U0001F77F"  # alchemical symbols
                               u"\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
                               u"\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
                               u"\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
                               u"\U0001FA00-\U0001FA6F"  # Chess Symbols
                               u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
                               u"\U00002702-\U000027B0"  # Dingbats
                               u"\U000024C2-\U0001F251" 
                               "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    
    # 멘션 제거
    mention_pattern = re.compile(r'@\S+')
    text = mention_pattern.sub(r'', text)
    
    # "@" 문자만 제거
    text = text.replace('@', '')
    
    # URL 제거
    url_pattern = re.compile(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+')
    text = url_pattern.sub(r'', text)

    # HTML Entities 제거
    html_entity_pattern = re.compile(r'&[a-zA-Z]+;')
    text = html_entity_pattern.sub(r'', text)
    
    # 특수문자 제거
    special_char_pattern = re.compile(r'[\^_\+\/,\(\):*]')
    text = special_char_pattern.sub('', text)
    # 한글 제거
    korean_char_pattern = re.compile(r'[가-힣]+')
    text = korean_char_pattern.sub(r'', text)
    
    # "(cont.)" 제거
    cont_pattern = re.compile(r'\(cont\.\)\s*|\s*\(cont\.\)')
    text = cont_pattern.sub(r'', text)

    # 줄바꿈 제거
    text = text.replace('\n', ' ')

    # "fvck", "fuck", "porn", "pukimak", "sial", "sialan", "bajingan" 제거
    explicit_words_pattern = re.compile(r'fvck|fuck|porn|pukimak|sial|sialan|bajingan', flags=re.IGNORECASE)
    text = explicit_words_pattern.sub('', text)
    
    # "["와 "]" 특수 문자 제거
    text = text.replace('[', '').replace(']', '')

    # 연속된 "#" 기호를 공백으로 대체
    text = re.sub(r'#\s+#', ' ', text)
    
    return text



df['Text']=df['Text'].apply(clean_text)


# 'Text' 컬럼 중심으로 중복 제거
df.drop_duplicates(subset='Text', inplace=True)
df.reset_index(drop=True, inplace=True)

df.head(10)

<br>
<br>
<br>

# Tokenizing

In [ ]:
import spacy
import stanza
import pandas as pd
from tqdm import tqdm

# 스파이시 영어 모델 로드
nlp = spacy.load('en_core_web_sm')

# 스탄자 인도네시아어 모델 로드
stanza_id = stanza.Pipeline('id', processors='tokenize')

In [ ]:
def tokenizing(df):
    new_rows = []
    max_length = 5
    for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Processing rows"):
        article_text = row['Text']
        doc = nlp(article_text)
        sen_id = 1  # 문장 ID를 1로 초기화


        for sentence in doc.sents:  # spacy에서는 .sents로 문장에 접근
            stanza_doc = stanza_id(sentence.text)
            tokens = [token.text for sent in stanza_doc.sentences for token in sent.tokens]
            
            # 문장, 토큰, 문장길이, Sen_ID 컬럼 추가
            new_row = row.copy()
            new_row['Sentence'] = sentence.text
            new_row['Token'] = tokens
            new_row['Sen_len'] = len(tokens)
            new_row['Sen_ID'] = f"{row['Doc_ID']}_sen{sen_id:06d}"
            
            new_rows.append(new_row)
            sen_id += 1  # 문장 ID 증가


    # 리스트 기호 없이 단어 단위로 토큰화
    new_df = pd.DataFrame(new_rows)
    new_df['Pub_Type']='SNS'
    new_df['Tokenized_Sentence']= new_df['Token'].apply(' '.join)

    # 'Sentence' 컬럼 중심으로 중복 제거
    new_df.drop_duplicates(subset='Sentence', inplace=True)
    new_df.reset_index(drop=True, inplace=True)

    # 공식 제출용 컬럼명으로 컬럼명 변환
    new_df.rename(columns={'Sen_len': 'Word_Count'}, inplace=True)
    
    new_df = new_df[['Doc_ID', 'Filename', 'Title', 'Pub_Type', 'Pub_Subj', 'Pub_Date', 'Col_Date', 'Sen_ID', 'Word_Count', 'Text', 'Sentence','Tokenized_Sentence' , 'Token']]
    new_df = new_df[new_df['Token'].apply(lambda x: len(x) > max_length)]
    return new_df


In [ ]:
# 토큰화 완료된 최종 데이터프레임
df = tokenizing(df)
df.head(10)

In [ ]:
df.reset_index(drop=True, inplace=True)
df.sort_values(by='Sen_ID', inplace=True)
df

In [ ]:
df['Word_Count'].sum()

<br>
<br>

# 데이터 저장

In [ ]:
# 폴더 이름 지정
folder_name = '원시데이터/twitterdata_all'

# 폴더가 없다면 생성
if not os.path.exists(folder_name):
    os.makedirs(folder_name)

df.to_csv(f"{folder_name}/{current_date}_twitter_{keyword}.csv", index = False, encoding = "utf-8-sig")